# 08 — Train the recorded models in Colab

Prepare the pinned ProvICS inputs, build the window features, fit the primary
Isolation Forest models, and save scores and model bundles in visible steps.
Optional cells fit supervised classifiers and recording-membership probes.
Notebooks 00–07 display and check the saved paper results.

**Execution status:** the training cells are unexecuted in this release.
Downloads and training are enabled explicitly; every step runs in the foreground.

The fitting functions, hyperparameters, preprocessing arithmetic, and corrected
M2 merge logic retain the original source definitions. Software checks cover
source equivalence, imports, path handling, and output safety. A full training
run is a separate validation step.

You will need the full repository checkout and access to the pinned ProvICS
dataset under its upstream terms. Raw data are not redistributed here. The eight
required files total about 13.9 GB; the large PCAP files are not needed. CPU is
sufficient for these classical models; raw provenance extraction can require
substantial RAM. Start with one 60-second resolution.

## 1. Locate the repository

Copy the full checkout to `MyDrive/provics-recording-aware-evaluation` first.
Once the repository is public, the README's clone command is another option.
Do not put GitHub or Hugging Face tokens in a saved notebook.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    default_root = Path('/content/drive/MyDrive/provics-recording-aware-evaluation')
else:
    default_root = Path.cwd()
    if not (default_root / 'configs/evaluation_settings.json').is_file():
        default_root = default_root.parent

ROOT = Path(os.environ.get('PROVICS_ROOT', default_root)).resolve()
assert (ROOT / 'configs/evaluation_settings.json').is_file(), 'Copy the full repository first'
os.environ['PROVICS_ROOT'] = str(ROOT)
sys.path.insert(0, str(ROOT))
print('Repository folder:', ROOT.name)

## 2. Install the training dependencies, only if you choose to

Change the flag below in a fresh Colab runtime to install the optional training
requirements. The numerical versions are recorded from the original Colab
environment; they are not the replay notebook's newer package versions.
If Colab asks for a restart, restart and rerun the setup cells before continuing.

In [ ]:
INSTALL_TRAINING_DEPS = False

if INSTALL_TRAINING_DEPS:
    get_ipython().run_line_magic('pip', f'install -r "{ROOT / "requirements-training.txt"}"')
else:
    print('Installation skipped. Set INSTALL_TRAINING_DEPS=True if needed.')

## 3. Choose which steps to run

Leave all action flags `False` to read this notebook without downloading,
preparing data, or training. Enable each stage only when you reach it.
The recorded primary fit uses seeds 42–46; no tuning or model search is added.
Change `WINDOW_S` to 5, 10, or 30 only to run that recorded resolution separately.

In [ ]:
ALLOW_DOWNLOADS = False
PREPARE_INPUTS = False
ALLOW_TRAINING = False
RUN_SUPERVISED = False
RUN_RECORDING_PROBE = False

WINDOW_S = 60
SEEDS = (42, 43, 44, 45, 46)

import json
import numpy as np
import pandas as pd
from IPython.display import display
from src.training.workspace import TrainingWorkspace
from src.training import inputs
from src.evaluation.replay import load_settings, verify_artifacts

workspace = TrainingWorkspace(ROOT / 'training_workspace', repository=ROOT)
settings = load_settings()
assert WINDOW_S in settings['windowing']['durations_seconds']
assert list(SEEDS) == settings['detector']['seeds']
original_environment = json.loads((ROOT / 'configs/training_source_manifest.json').read_text())['original_training_environment']
display(pd.DataFrame(original_environment.items(), columns=['Original package', 'Recorded version']))
display(pd.DataFrame([{'Step':'Download raw files', 'Enabled':ALLOW_DOWNLOADS},
    {'Step':'Prepare features', 'Enabled':PREPARE_INPUTS},
    {'Step':'Fit primary detectors', 'Enabled':ALLOW_TRAINING},
    {'Step':'Fit supervised comparisons', 'Enabled':ALLOW_TRAINING and RUN_SUPERVISED},
    {'Step':'Fit recording probes', 'Enabled':ALLOW_TRAINING and RUN_RECORDING_PROBE}]))

## 4. Check the saved release and inspect the download list

This cell reads existing files only. SHA-256 protects the saved paper artifacts.
For source files, the older download manifest uses SHA-256 for LFS objects and
Git-blob SHA-1 for ordinary files; the downloader checks the appropriate kind.

In [ ]:
canonical_before = verify_artifacts()
assert all(item['status'] == 'ok' for item in canonical_before)
print('Protected artifacts:', len(canonical_before))

inventory = inputs.input_inventory(workspace)
display(inventory[['path', 'size', 'sha']])
print('Required source files:', len(inventory))
print('Download size (decimal GB):', round(inventory['size'].sum() / 1e9, 2))

## 5. Download and verify the raw inputs

This is an explicit large download, pinned to the paper's dataset revision.
Existing verified files are reused. A wrong-size or wrong-checksum file stops
the workflow; it is not silently replaced. Progress appears under this cell.

In [ ]:
if ALLOW_DOWNLOADS:
    inputs.download_inputs(workspace)
else:
    print('Download skipped. No remote data were fetched.')

## 6. Inspect the metadata and prepare physical features (M4)

The released score grids define row order and labels. They are not the older
inventory grid. The original physical builder produces **199 features** from
22 process variables: nine summaries per variable, plus sample count. It does
not replace the original representation with 22 means.

In [ ]:
meta_a, meta_b, events = inputs.load_metadata(workspace, WINDOW_S)
display(pd.DataFrame([{'Recording':'attack22h', 'Windows':len(meta_a)},
    {'Recording':'benign48h', 'Windows':len(meta_b)}]))
if PREPARE_INPUTS:
    inputs.prepare_physical(workspace, WINDOW_S)
else:
    print('Physical feature preparation skipped.')

## 7. Prepare supervisory-host provenance (M1)

This cell parses only M1. Semantic counters replace raw identifiers. Category
vocabularies will be fitted later using the benign fitting partition only.
The parser prints progress here; the cell does not launch another process.

In [ ]:
if PREPARE_INPUTS:
    inputs.prepare_cyber(workspace, 'M1', WINDOW_S)
else:
    print('M1 extraction skipped.')

## 8. Prepare controller provenance (M2)

The corrected extractor merges repeated fragments of the same time window
before computing features. It does not overwrite an earlier fragment.

In [ ]:
if PREPARE_INPUTS:
    inputs.prepare_cyber(workspace, 'M2', WINDOW_S)
else:
    print('M2 extraction skipped.')

## 9. Prepare decoded Modbus traffic (M3)

M3 uses protocol counters, function codes, register-reference statistics, and
timing. The released source has no register-value payload field. Raw addresses,
ports, transaction IDs, and timestamps are not passed to a detector.

In [ ]:
if PREPARE_INPUTS:
    inputs.prepare_cyber(workspace, 'M3', WINDOW_S)
else:
    print('M3 extraction skipped.')

## 10. Assemble and inspect the model inputs

Category vocabularies use the first 70% of the separate benign recording only.
Availability is stored as a mask, not a model feature. The schema guard rejects
identifier and label fields. Feature names and order must exactly match the
released feature list; a mismatch stops the workflow rather than retuning it.

Prepared-input manifests must be complete and their checksums must match. If a
preparation cell was interrupted, use a fresh workspace or inspect that partial
block yourself; the notebook will not delete or overwrite it for you.

In [ ]:
prepared = None
if PREPARE_INPUTS or ALLOW_TRAINING:
    prepared = inputs.load_prepared_views(workspace, WINDOW_S)
    display(pd.DataFrame([{'Modality':name, 'Features':a.shape[1],
        'Attack windows':len(a), 'Benign windows':len(b),
        'Attack availability':float(ava.mean()), 'Benign availability':float(avb.mean())}
        for name,(a,b,ava,avb) in prepared['views'].items()]))
else:
    print('Input assembly skipped; no prepared or raw data are required to read the notebook.')

## 11. Understand the fit and create an isolated run folder

The primary estimator is `IsolationForest(n_estimators=300, max_samples=256,
bootstrap=False)` with the recorded seed. Training-median imputation and
`StandardScaler` use benign fitting rows only. The score is `-score_samples`:
higher values mean more anomalous.

The primary reference is **retrospective**: the supplied benign capture occurs
after the attack capture in calendar time. The first 70% of benign windows is
also reused for score calibration; the last 30% is cross-recording background.
This is not the separate action-screened historical-reference audit.

All new scores, models, and summaries go under a unique
`training_workspace/runs/...` folder. Canonical results are never overwritten.

In [ ]:
from src.training.isolation_forest import score_view, IF_PARAMS
from src.training.model_bundle import bundle_primary, score_bundle

run = None
run_hashes = {}
trained_scores = {}
print('Primary parameters:', IF_PARAMS)
print('Seeds:', SEEDS)
if ALLOW_TRAINING:
    assert prepared is not None, 'Prepare and inspect the inputs first'
    run = workspace.new_run()
    workspace.write_json(run / 'run_settings.json', {
        'window_s':WINDOW_S, 'seeds':SEEDS, 'if_params':IF_PARAMS,
        'reference':settings['primary_reference'],
        'dataset_revision':settings['dataset']['pinned_revision'],
        'original_environment':original_environment,
        'execution_environment':{'numpy':np.__version__, 'pandas':pd.__version__,
            'sklearn':__import__('sklearn').__version__},
        'scope':'new opt-in training run; not canonical paper results'})
    print('New run:', run.as_posix())
else:
    print('Run folder not created. No detector has been fitted.')

## 12. Fit the primary models, one fit at a time

This is the primary training cell. `score_view()` contains the original
`IsolationForest(...).fit(...)` call. One resolution contains 4 modalities x 5
seeds = 20 fits. Progress is printed before and after each fit.

The original fitting-row convention is retained, including median-imputed
missing feature rows in the benign fit. Availability masks restrict reported
per-view evaluation. The portability work does not silently alter that policy.
Each saved model bundle includes its feature order, training median, scaler,
estimator, and library version. Only load bundles you created or trust.

In [ ]:
if ALLOW_TRAINING:
    for modality, (xa, xb, available_a, available_b) in prepared['views'].items():
        for seed in SEEDS:
            print(f'Fitting {modality}, window={WINDOW_S}s, seed={seed}', flush=True)
            score_a, score_b, _, estimator = score_view(xa, xb, prepared['benign_fit'], seed)
            bundle = bundle_primary(estimator, xb, prepared['benign_fit'])
            np.testing.assert_allclose(score_bundle(bundle, xa), score_a, rtol=0, atol=1e-12)
            name = f'{modality}_w{WINDOW_S}_s{seed}'
            run_hashes[name+'.npz'] = workspace.write_npz(run / (name+'.npz'),
                attack=score_a, benign=score_b,
                availability_attack=available_a, availability_benign=available_b)
            run_hashes[name+'.joblib'] = workspace.write_model(run / (name+'.joblib'), bundle)
            trained_scores[(modality, seed)] = (score_a, score_b)
            print('Saved scores and model:', name, flush=True)
else:
    print('Primary training skipped. Existing scores and results are unchanged.')

## 13. Inspect the new scores under the two background comparisons

The positives remain fixed. Cross-recording negatives come from the last 30%
of the benign recording; same-recording negatives come from the attack capture,
outside labeled actions and the fixed +/-300-second buffer. This cell creates
a new summary only if the training cell above was explicitly run.

In [ ]:
from src.evaluation.metrics import cross_within

if ALLOW_TRAINING and trained_scores:
    rows = []
    for (modality, seed), (score_a, score_b) in trained_scores.items():
        _, _, available_a, available_b = prepared['views'][modality]
        benign_eval = prepared['benign_eval']
        comparison = cross_within(score_a[prepared['action'] & available_a],
            score_b[benign_eval][available_b[benign_eval]],
            score_a[prepared['background'] & available_a])
        rows.append({'window_s':WINDOW_S, 'modality':modality, 'seed':seed, **comparison})
    primary_summary = pd.DataFrame(rows)
    display(primary_summary)
    run_hashes['primary_window_summary.csv'] = workspace.write_csv(
        run / 'primary_window_summary.csv', primary_summary)
else:
    print('No new summary computed. Saved paper results remain in results/paper_tables/.')

## 14. Optional supervised comparisons

The original logistic regression, random forest, and MLP functions use earlier
attack campaigns as positives and the separate benign fitting rows as negatives.
They evaluate the next campaign against both negative populations. This retains
the source's MLP early-stopping rule and fitting/evaluation conventions; no new
hyperparameter search or chronological guarantee for the benign capture is added.
This cell remains disabled unless both training flags are enabled.

In [ ]:
from src.training.supervised import supervised_forward

if ALLOW_TRAINING and RUN_SUPERVISED:
    tables = []
    for modality, (xa, xb, _, _) in prepared['views'].items():
        for seed in SEEDS:
            print('Supervised comparison:', modality, seed, flush=True)
            tables.append(supervised_forward(xa, xb, prepared['meta_attack'],
                prepared['events'], prepared['campaign'], prepared['action'],
                prepared['background'], prepared['benign_fit'], prepared['benign_eval'],
                modality, WINDOW_S, seed))
    supervised_summary = pd.concat(tables, ignore_index=True)
    display(supervised_summary)
    run_hashes['supervised_comparison.csv'] = workspace.write_csv(
        run / 'supervised_comparison.csv', supervised_summary)
else:
    print('Supervised fitting skipped.')

## 15. Optional recording-membership probes

This target is recording membership, not attacks. Each recording is split in
time order: its first 70% for fitting and last 30% for testing. Only pre-action
attack-recording windows are used; available-view masks are kept outside the
features. The original model families are LR, RF, and MLP.

In [ ]:
from src.training.session_probe import session_probe

if ALLOW_TRAINING and RUN_RECORDING_PROBE:
    tables = []
    for modality, (xa, xb, available_a, available_b) in prepared['views'].items():
        pre = xa.loc[prepared['pre_action'] & available_a].reset_index(drop=True)
        benign = xb.loc[available_b].reset_index(drop=True)
        for seed in SEEDS:
            print('Recording probe:', modality, seed, flush=True)
            table, _, _ = session_probe(pre, benign, seed)
            table['modality'] = modality
            table['window_s'] = WINDOW_S
            tables.append(table)
    recording_summary = pd.concat(tables, ignore_index=True)
    display(recording_summary)
    run_hashes['recording_probe.csv'] = workspace.write_csv(run / 'recording_probe.csv', recording_summary)
else:
    print('Recording-probe fitting skipped.')

## 16. Verify that the paper's artifacts are unchanged

Compare the canonical file checksums before and after the workflow. New training
outputs stay in their own run directory, separate from the saved paper tables.
Use the recorded dependency versions when comparing a new run with saved results.

In [ ]:
canonical_after = verify_artifacts()
assert all(item['status'] == 'ok' for item in canonical_after)
if ALLOW_TRAINING:
    workspace.write_json(run / 'output_checksums.json', run_hashes)
    print('New run outputs:', run.as_posix())
else:
    print('No training outputs were created by this notebook.')
print('Canonical artifact checksums still match:', len(canonical_after))

## Workflow guide

- **Prepare and train:** use this notebook for corrected provenance/Modbus extraction,
  physical features, primary IF models, supervised comparisons, and recording probes.
- **Inspect saved results:** notebooks 00–07 contain saved cell outputs; notebook 04
  covers fusion and alarms, notebook 05 covers temporal/calibration summaries,
  and notebook 06 covers the HAI analysis.
- **Align new scores:** input preparation builds per-view availability masks. The
  original seed-42 score archive contains scores and paired row metadata, without
  those masks; use the saved tables for its modality-specific AUC summaries.
- **Keep runs separate:** new features, scores, and model bundles are saved under
  `training_workspace/`; the paper's released tables and figures remain unchanged.